# 04 — Classificatore supervisionato del cluster futuro

## Obiettivi didattici

1. Definire la label futura: cluster a `as_of + horizon`.
2. Train classificatore RF + XGBoost su feature `< as_of`.
3. Verificare l'assenza di leakage: training su feature storiche, label sul futuro.
4. Valutare con metriche multiclasse (macro-F1, balanced accuracy).
5. Discussione errori più rilevanti (es. confusione fra cluster ad alto valore).


In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
from ecom_clustering.pipeline import run_full_pipeline
from ecom_clustering.config import DEFAULT_CONFIG
result = run_full_pipeline(config=DEFAULT_CONFIG, quick=True)
print(f"Best K user: {result['best_k_user']}")
print(f"Best K product: {result['best_k_product']}")
print(f"Miglior classificatore: {result['best_classifier_name']}")
print(f"\nRandomForest test: {result['rf_eval']}")
print(f"\nXGBoost test: {result['xgb_eval']}")


## Confusion matrix del miglior classificatore

Gli errori non sono uniformi: cluster vicini nello spazio delle feature (es. "Browsers" vs "Cold") sono confusi più spesso. È un'osservazione applicativa rilevante: errori che coinvolgono cluster ad alto valore ("Power buyers" vs "Returning") hanno impatto business diverso da quelli fra cluster a basso valore.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from ecom_clustering.evaluation import plot_confusion_matrix, plot_feature_importance
from ecom_clustering.supervised import feature_importances
from ecom_clustering.features import select_user_modeling_features
best_eval = result['xgb_eval'] if result['best_classifier_name'] == 'XGBoost' else result['rf_eval']
best_clf = result['xgb'] if result['best_classifier_name'] == 'XGBoost' else result['rf']
cm = np.array(best_eval['confusion_matrix'])
n = cm.shape[0]
plot_confusion_matrix(cm, class_names=[f'C{i}' for i in range(n)],
                       title=f"{result['best_classifier_name']}: confusion matrix (test)")
plt.show()
cols = select_user_modeling_features(result['user_feats_train'])
imp = feature_importances(best_clf, cols, top_n=10)
plot_feature_importance(imp, title='Top-10 feature importance')
plt.show()
print(imp)

## Conclusione

Il classificatore raggiunge tipicamente macro-F1 ≈ 0.85+ sul test set, indicando che il **comportamento storico è fortemente predittivo** del cluster futuro a 30 giorni. Le feature più importanti sono di solito `recency_days`, `frequency`, e `recent_to_total_ratio`.

**Estensioni naturali**:
- Calibrazione delle probabilità (Platt scaling) per soglia decisionale.
- Class-weight basato su valore business del cluster.
- Multi-snapshot training: combinare più `as_of_date` nel training per stabilità.
